In [1]:
!pip install mealpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.1/149.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.9/397.9 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 77.8 MB/s eta 0:00:00


In [2]:
import csv
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
from PIL import Image

In [3]:
def load_flatten_images(real_dir, fake_dir, img_size=(224, 224), max_per_class=5000):
    X, y = [], []
    for label, folder in [(0, real_dir), (1, fake_dir)]:
        files = os.listdir(folder)[:max_per_class]
        for fname in tqdm(files, desc=f"Loading {folder}"):
            path = os.path.join(folder, fname)
            img = Image.open(path).convert("RGB").resize(img_size)  
            X.append(np.array(img).transpose(2, 0, 1).flatten())    
            y.append(label)
    return np.array(X), np.array(y)

In [4]:
real_dir = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/real"
fake_dir = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/fake"

X_fs, y_fs = load_flatten_images(real_dir, fake_dir)

Loading /kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/real: 100%|██████████| 5000/5000 [00:46<00:00, 107.79it/s]
Loading /kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train/fake: 100%|██████████| 5000/5000 [00:43<00:00, 115.81it/s]


In [5]:
from sklearn.feature_selection import SelectKBest, f_classif
import joblib

selector = SelectKBest(score_func=f_classif, k=5000) 
X_new = selector.fit_transform(X_fs, y_fs)

joblib.dump(selector, "selector_kbest.pkl")

['selector_kbest.pkl']

In [6]:
from sklearn.ensemble import RandomForestClassifier
from mealpy.evolutionary_based import GA
from mealpy import FloatVar

def fitness_func(solution):
    mask = np.array(solution) > 0.5
    if np.sum(mask) == 0:
        return 1.0  

    X_selected = X_new[:, mask]
    X_train, X_test, y_train, y_test = train_test_split(X_selected, y_fs, test_size=0.2, stratify=y_fs, random_state=42)

    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)
    preds = clf.predict(X_test)
    acc = accuracy_score(y_test, preds)

    return 1 - acc

num_features = X_new.shape[1]
problem = {
    "bounds": FloatVar(lb=(0.0,) * num_features, ub=(1.0,) * num_features, name="feature_selector"),
    "obj_func": fitness_func,
    "minmax": "min",
}

model = GA.BaseGA(epoch=30, pop_size=20, pc=0.9, pm=0.05)
g_best = model.solve(problem, mode="thread")

In [7]:
import pandas as pd

save_dir = "mealpy_feature_selection"
os.makedirs(save_dir, exist_ok=True)
hist_dir = os.path.join(save_dir, "history_logs")
os.makedirs(hist_dir, exist_ok=True)

final_mask = np.array(g_best.solution) > 0.5
np.save(os.path.join(save_dir, "best_feature_mask.npy"), final_mask)
np.savetxt(os.path.join(save_dir, "selected_indices.csv"), np.where(final_mask)[0], fmt="%d")
with open(os.path.join(save_dir, "best_fitness.txt"), "w") as f:
    f.write(f"Fitness = {g_best.target.fitness:.6f}\nAccuracy = {1 - g_best.target.fitness:.4f}")

save_dir = "mealpy_feature_selection/history_logs_csv"
os.makedirs(save_dir, exist_ok=True)

def save_list_as_csv(data_list, filename):
    df = pd.DataFrame(data_list)
    df.to_csv(filename, index=False)

save_list_as_csv(model.history.list_global_best, os.path.join(save_dir, "global_best.csv"))
save_list_as_csv(model.history.list_current_best, os.path.join(save_dir, "current_best.csv"))
save_list_as_csv(model.history.list_global_worst, os.path.join(save_dir, "global_worst.csv"))
save_list_as_csv(model.history.list_current_worst, os.path.join(save_dir, "current_worst.csv"))
save_list_as_csv(model.history.list_epoch_time, os.path.join(save_dir, "epoch_time.csv"))
save_list_as_csv(model.history.list_global_best_fit, os.path.join(save_dir, "global_best_fit.csv"))
save_list_as_csv(model.history.list_current_best_fit, os.path.join(save_dir, "current_best_fit.csv"))
save_list_as_csv(model.history.list_diversity, os.path.join(save_dir, "diversity.csv"))
save_list_as_csv(model.history.list_exploration, os.path.join(save_dir, "exploration.csv"))
save_list_as_csv(model.history.list_exploitation, os.path.join(save_dir, "exploitation.csv"))

print(f"Accuracy: {1 - g_best.target.fitness:.4f}")
print(f"Selected {final_mask.sum()} features: {np.where(final_mask)[0].tolist()}")
print(f"All results saved to: {save_dir}")

Accuracy: 0.7430
Selected 2421 features: [2, 3, 6, 8, 11, 14, 15, 16, 17, 19, 20, 21, 22, 23, 24, 25, 27, 30, 31, 32, 34, 39, 40, 43, 44, 45, 46, 47, 48, 50, 54, 58, 61, 62, 63, 66, 67, 70, 71, 75, 76, 77, 78, 86, 87, 89, 93, 96, 98, 102, 103, 104, 105, 107, 112, 115, 118, 119, 122, 123, 125, 127, 128, 129, 133, 134, 136, 137, 139, 142, 145, 146, 152, 154, 156, 158, 159, 161, 163, 164, 168, 169, 170, 172, 173, 176, 178, 180, 183, 184, 186, 187, 189, 195, 197, 201, 202, 206, 207, 208, 209, 210, 212, 213, 214, 216, 217, 218, 220, 224, 226, 228, 230, 231, 232, 233, 234, 235, 240, 243, 244, 245, 246, 248, 249, 252, 254, 255, 256, 258, 261, 263, 265, 266, 267, 268, 271, 273, 274, 275, 277, 279, 281, 286, 292, 296, 298, 299, 302, 303, 306, 308, 309, 310, 313, 317, 318, 319, 320, 322, 325, 326, 328, 331, 333, 334, 338, 343, 345, 349, 352, 354, 356, 357, 358, 360, 361, 362, 364, 366, 367, 368, 369, 372, 374, 376, 377, 378, 379, 380, 381, 382, 389, 390, 393, 400, 402, 403, 404, 408, 409, 410, 4